In [22]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import numpy as np

In [23]:

model=load_model('model.h5')
with open('label_enc_gender.pkl','rb') as file:
  label_enc_gender=pickle.load(file)
with open('onehot_enc_geo.pkl','rb') as file:
  onehot_enc_geo=pickle.load(file)
with open('scaler.pkl','rb') as file:
  scaler=pickle.load(file)

In [24]:
#example input data
input_data={
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [25]:
geo_encoded=onehot_enc_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_enc_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [26]:
##JSON input to Dataframe
input_data=pd.DataFrame([input_data])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [27]:
#encode categorical values
input_data['Gender']=label_enc_gender.transform(input_data['Gender'])
input_data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [28]:
input_data=pd.concat([input_data.drop('Geography',axis=1),geo_encoded_df],axis=1)
input_data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [29]:
#scaling the input data
input_scaled=scaler.transform(input_data)
input_scaled

array([[-0.47154541,  0.90911166,  0.09477172, -0.69844549, -0.29010416,
         0.80510537,  0.63367318,  0.95214374, -0.84805047,  0.98019606,
        -0.57581067, -0.56349184]])

In [30]:
#prediction
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step


array([[0.03191087]], dtype=float32)

In [33]:
prediction_proba=prediction[0][0]
print(prediction_proba)

0.031910866


In [34]:
if prediction_proba>0.5:
  print('The customer is likely to churn')
else:
  print('The customer is not likely to churn')

The customer is not likely to churn
